# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [18]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display


In [19]:
# import google.generativeai as genai

In [20]:
from google import genai
from google.genai import types

In [21]:
from dotenv import load_dotenv
load_dotenv()

CONFIG = {
    "company_name": "HuggingFace",
    "url": "https://huggingface.co",
    "api_key": os.getenv("GEMINI_API_KEY"),
    "model": "gemini-2.0-flash"
}

In [22]:

headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """
    A utility class to represent a Website that we have scraped, now with links
    """

    def __init__(self, url):
        try:
            self.url = url
            response = requests.get(url, headers=headers)
            self.body = response.content
            soup = BeautifulSoup(self.body, 'html.parser')
            self.title = soup.title.string if soup.title else "No title found"
            if soup.body:
                for irrelevant in soup.body(["script", "style", "img", "input"]):
                    irrelevant.decompose()
                self.text = soup.body.get_text(separator="\n", strip=True)
            else:
                self.text = ""
            links = [link.get('href') for link in soup.find_all('a')]
            self.links = [link for link in links if link]
        except:
            print("website error")
    def get_contents(self):
        try:
            return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"
        except:
            return ""

In [23]:
ed = Website(CONFIG["url"])
ed.links

['/',
 '/models',
 '/datasets',
 '/spaces',
 '/posts',
 '/docs',
 '/enterprise',
 '/pricing',
 '/login',
 '/join',
 '/spaces',
 '/models',
 '/HiDream-ai/HiDream-I1-Full',
 '/agentica-org/DeepCoder-14B-Preview',
 '/moonshotai/Kimi-VL-A3B-Thinking',
 '/moonshotai/Kimi-VL-A3B-Instruct',
 '/deepseek-ai/DeepSeek-V3-0324',
 '/models',
 '/spaces/enzostvs/deepsite',
 '/spaces/jamesliu1217/EasyControl_Ghibli',
 '/spaces/bytedance-research/UNO-FLUX',
 '/spaces/Efficient-Large-Model/SanaSprint',
 '/spaces/HiDream-ai/HiDream-I1-Dev',
 '/spaces',
 '/datasets/nvidia/OpenCodeReasoning',
 '/datasets/openai/mrcr',
 '/datasets/nvidia/Llama-Nemotron-Post-Training-Dataset',
 '/datasets/divaroffical/real_estate_ads',
 '/datasets/openai/graphwalks',
 '/datasets',
 '/join',
 '/pricing#endpoints',
 '/pricing#spaces',
 '/pricing',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/allenai',
 '/facebook',
 '/amazon',
 '/google',
 '/Intel',
 '/micro

## First step: Have GPT-4o-mini figure out which links are relevant

### Use a call to gpt-4o-mini to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [24]:
link_system_prompt = "You are provided with a list of links found on a webpage. \
You are able to decide which of the links would be most relevant to include in a brochure about the company, \
such as links to an About page, or a Company page, or Careers/Jobs pages.\n"
link_system_prompt += "You should respond in JSON as in this example:"
link_system_prompt += """
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [25]:
print(link_system_prompt)

You are provided with a list of links found on a webpage. You are able to decide which of the links would be most relevant to include in a brochure about the company, such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}



In [26]:
def get_links_user_prompt(website):
    user_prompt = f"Here is the list of links on the website of {website.url} - "
    user_prompt += "please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. \
Do not include Terms of Service, Privacy, email links.\n"
    user_prompt += "Links (some might be relative links):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [27]:
print(get_links_user_prompt(ed))

Here is the list of links on the website of https://huggingface.co - please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. Do not include Terms of Service, Privacy, email links.
Links (some might be relative links):
/
/models
/datasets
/spaces
/posts
/docs
/enterprise
/pricing
/login
/join
/spaces
/models
/HiDream-ai/HiDream-I1-Full
/agentica-org/DeepCoder-14B-Preview
/moonshotai/Kimi-VL-A3B-Thinking
/moonshotai/Kimi-VL-A3B-Instruct
/deepseek-ai/DeepSeek-V3-0324
/models
/spaces/enzostvs/deepsite
/spaces/jamesliu1217/EasyControl_Ghibli
/spaces/bytedance-research/UNO-FLUX
/spaces/Efficient-Large-Model/SanaSprint
/spaces/HiDream-ai/HiDream-I1-Dev
/spaces
/datasets/nvidia/OpenCodeReasoning
/datasets/openai/mrcr
/datasets/nvidia/Llama-Nemotron-Post-Training-Dataset
/datasets/divaroffical/real_estate_ads
/datasets/openai/graphwalks
/datasets
/join
/pricing#endpoints
/pricing#spaces
/pricing
/enterprise
/enterpris

In [28]:
def get_links():
    client = genai.Client(api_key=CONFIG["api_key"])
    website = Website(CONFIG["url"])
    prompt = f"{link_system_prompt}\n\n{get_links_user_prompt(website)}"
    response = client.models.generate_content(
        model=CONFIG["model"],
        contents=prompt
    )
    try:
        return json.loads(response.text)
    except json.JSONDecodeError:
        print("Warning: Invalid JSON response:", response.text)
        return {"links": []}
    except Exception as e:
        print(f"Error generating content: {e}")
        return {"links": []}

In [29]:
# Anthropic has made their site harder to scrape, so I'm using HuggingFace..

huggingface = Website(CONFIG["url"])
huggingface.links

['/',
 '/models',
 '/datasets',
 '/spaces',
 '/posts',
 '/docs',
 '/enterprise',
 '/pricing',
 '/login',
 '/join',
 '/spaces',
 '/models',
 '/HiDream-ai/HiDream-I1-Full',
 '/agentica-org/DeepCoder-14B-Preview',
 '/moonshotai/Kimi-VL-A3B-Thinking',
 '/moonshotai/Kimi-VL-A3B-Instruct',
 '/deepseek-ai/DeepSeek-V3-0324',
 '/models',
 '/spaces/enzostvs/deepsite',
 '/spaces/jamesliu1217/EasyControl_Ghibli',
 '/spaces/bytedance-research/UNO-FLUX',
 '/spaces/Efficient-Large-Model/SanaSprint',
 '/spaces/HiDream-ai/HiDream-I1-Dev',
 '/spaces',
 '/datasets/nvidia/OpenCodeReasoning',
 '/datasets/openai/mrcr',
 '/datasets/nvidia/Llama-Nemotron-Post-Training-Dataset',
 '/datasets/divaroffical/real_estate_ads',
 '/datasets/openai/graphwalks',
 '/datasets',
 '/join',
 '/pricing#endpoints',
 '/pricing#spaces',
 '/pricing',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/allenai',
 '/facebook',
 '/amazon',
 '/google',
 '/Intel',
 '/micro

## Second step: make the brochure!

Assemble all the details into another prompt to GPT4-o

In [30]:
def get_all_details():
    """Fetch the landing page and relevant links' content."""
    try:
        result = "Landing page:\n"
        result += Website(CONFIG["url"]).get_contents()
        links = get_links()
        print("Found links:", links)
        for link in links["links"]:
            try:
                result += f"\n\n{link['type']}\n"
                result += Website(link["url"]).get_contents()
            except Exception as e:
                print(f"Error processing link: {e}")
        return result
    except Exception as e:
        print(f"Error getting details: {e}")
        return f"Error analyzing website {CONFIG['url']}: {str(e)}"

In [31]:
system_prompt = """You are an assistant that analyzes the contents of several relevant pages from a company website 
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond ONLY in markdown format. Do not show your reasoning or analysis steps.
Include only the formatted brochure with details about company culture, customers and careers/jobs if available."""

In [32]:
def get_brochure_user_prompt():
    """Generate the user prompt for creating the brochure."""
    user_prompt = f"You are looking at a company called: {CONFIG['company_name']}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown. Mention any relevant links.\n"
    user_prompt += get_all_details()
    user_prompt = user_prompt[:5_000] 
    return user_prompt

In [33]:
def create_brochure():
    client = genai.Client(api_key=CONFIG["api_key"])
    prompts = system_prompt + "\n" + get_brochure_user_prompt()
    response = client.models.generate_content_stream(
        contents=prompts,
        model=CONFIG["model"]
    )
    result = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in response:
        if chunk.text:
            result += chunk.text
            result = result.replace("```", "").replace("markdown", "")
            update_display(Markdown(result), display_id=display_handle.display_id)
    with open("brochure.md", "w", encoding="utf-8") as f:
        f.write(result)

In [34]:
create_brochure()

{
    "links": [
        {"type": "about page", "url": "https://huggingface.co/huggingface"},
        {"type": "pricing page", "url": "https://huggingface.co/pricing"},
        {"type": "enterprise page", "url": "https://huggingface.co/enterprise"},
        {"type": "careers page", "url": "https://apply.workable.com/huggingface/"},
        {"type": "blog", "url": "https://huggingface.co/blog"}
    ]
}
```
Found links: {'links': []}



# Hugging Face: The AI Community Building the Future

**Welcome to the leading platform for machine learning collaboration.**

[Hugging Face Website](https://huggingface.co/)

## About Us

Hugging Face is the home of Machine Learning, a collaborative platform where the ML community can create, discover, and collaborate on models, datasets, and applications. We're building the future of AI, together.

## What We Offer

*   **Models:** Explore over 1 million pre-trained models for various tasks (text, vision, audio, and more). [Browse Models](https://huggingface.co/models)

*   **Datasets:** Access over 250k datasets to train and evaluate your models. [Explore Datasets](https://huggingface.co/datasets)

*   **Spaces:** Discover and host ML applications. See trending applications and run demos instantly. [Discover Spaces](https://huggingface.co/spaces)

*   **Open Source Tools:** Leverage powerful open-source libraries:
    *   **Transformers:** State-of-the-art ML for PyTorch, TensorFlow, JAX.
    *   **Diffusers:** Cutting-edge Diffusion models in PyTorch.
    *   **Datasets:**  Access & share datasets for any ML tasks.

## For Businesses

**Accelerate your ML initiatives with our Enterprise solutions:**

*   **Compute:** Deploy on optimized Inference Endpoints or update your Spaces applications to a GPU in a few clicks.
*   **Enterprise:**  Enterprise-grade security, access controls, dedicated support and priority support to build AI.
    *   Single Sign-On
    *   Regions
    *   Audit Logs
    *   Resource Groups
    *   Private Datasets Viewer

More than 50,000 organizations are using Hugging Face!

## Community

Join a vibrant community of researchers, developers, and enthusiasts. Collaborate, share your work, and build your ML profile.

## Careers

We're always looking for talented individuals to join our team. Visit our [Jobs](https://huggingface.co/jobs) page for open positions.


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>